# 01 – Quickstart: ODYSSEY vs CIRRUS over Belgium

This notebook shows the basic building blocks of `opera_ds`:

1. download a few composites from the public OPERA archive (ORD bucket, CC BY 4.0, no credentials);
2. read them over Belgium + 200 km;
3. interpolate one ODYSSEY pair (2 km / 15 min) to 1 km / 5 min and compare with CIRRUS (1 km / 5 min).

It downloads about 10 MB into `data/` (or `$OPERA_DATA_DIR`) the first time and runs in under a minute.

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime as dt
from functools import partial

import matplotlib.pyplot as plt
import numpy as np

from opera_ds import DownscaleConfig, FrameSource, read_odim_composite
from opera_ds.odim_io import bbox_extent
from opera_ds.ord import download
from opera_ds.paths import DATA_DIR

DATA_DIR

## 1. Download

ODYSSEY and CIRRUS overlap in the public archive only on 1–5 July 2024, and on those days CIRRUS is missing at :00, :15, :30, :45.
We take the ODYSSEY pair 12:00 → 12:15 (plus one frame on each side, used for the motion field) and the CIRRUS frames at 12:05 and 12:10.

In [ ]:
T0 = dt.datetime(2024, 7, 2, 12, 0)
step = dt.timedelta(minutes=15)

odyssey = {t: download(t, "DBZH_QIND", DATA_DIR / "odyssey") for t in [T0 - step, T0, T0 + step, T0 + 2 * step]}
cirrus = {t: download(t, "DBZH", DATA_DIR / "cirrus") for t in [T0 + dt.timedelta(minutes=5), T0 + dt.timedelta(minutes=10)]}
odyssey

## 2. Read over Belgium + 200 km

`bbox_extent` returns a crop in metres from the upper-left corner of the OPERA grid, aligned to 2 km, so the 2 km and 1 km grids stay 2:1.
`read_odim_composite(..., extent=...)` reads only that window. Values are in dBZ; `coverage` is False outside radar range, `undetect` is True where there is coverage but no echo.

In [ ]:
BBOX_BEL = (2.5, 49.45, 6.45, 51.55)  # lon_min, lat_min, lon_max, lat_max

full = read_odim_composite(odyssey[T0])
extent = bbox_extent(full.where, BBOX_BEL, buffer_km=200)
reader = partial(read_odim_composite, extent=extent)

ody0 = reader(odyssey[T0])
cir = {t: reader(p) for t, p in cirrus.items()}
print("full ODYSSEY grid", full.values.shape, "-> crop", ody0.values.shape)
print("CIRRUS crop", next(iter(cir.values())).values.shape)

In [ ]:
def show(ax, frame_or_values, title, coverage=None, undetect=None):
    """dBZ map: no echo in white, outside radar coverage in grey."""
    if hasattr(frame_or_values, "values"):
        v, coverage, undetect = frame_or_values.values, frame_or_values.coverage, frame_or_values.undetect
    else:
        v = frame_or_values
    v = np.where(coverage & ~undetect & (v >= 0), v, np.nan)
    ax.imshow(np.where(coverage, np.nan, 1), cmap="Greys", vmin=0, vmax=3, interpolation="nearest")
    im = ax.imshow(v, cmap="turbo", vmin=0, vmax=55, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    return im


fig, axs = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
show(axs[0], ody0, f"ODYSSEY {T0:%H:%M} (2 km)")
im = show(axs[1], cir[T0 + dt.timedelta(minutes=5)], f"CIRRUS {T0 + dt.timedelta(minutes=5):%H:%M} (1 km)")
fig.colorbar(im, ax=axs, label="dBZ", shrink=0.8);

## 3. Interpolate to 1 km / 5 min

`FrameSource` reads and caches the ODYSSEY frames, estimates the Lucas-Kanade motion on the 2 km grid and returns a `PairInterpolator` for the pair (t0, t0 + 15 min).
`pair.get(alpha, method)` gives the field at t0 + alpha · 15 min on the 1 km grid, for `nearest`, `linear` (Eulerian) or `lk` (advection).

With the default labels (no time shift) alpha = 1/3 and 2/3 correspond to 12:05 and 12:10.

In [ ]:
cfg = DownscaleConfig()
src = FrameSource(odyssey, cfg, reader=reader)
pair = src.pair(T0)

alpha, t = 1 / 3, T0 + dt.timedelta(minutes=5)
fig, axs = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
for ax, method in zip(axs, ["nearest", "linear", "lk"]):
    val, cov, und = pair.get(alpha, method)
    show(ax, val, f"{method}, alpha = {alpha:.2f}", cov, und)
im = show(axs[3], cir[t], f"CIRRUS {t:%H:%M}")
fig.colorbar(im, ax=axs, label="dBZ", shrink=0.8);

## 4. A quick score

RMSE in dBZ on pixels that are wet in either field, at both interpolated times. Note that ODYSSEY and CIRRUS differ even without interpolation (see `docs/data.md`, *Known problems*), so only the differences between methods are meaningful here.

In [ ]:
for alpha, t in [(1 / 3, T0 + dt.timedelta(minutes=5)), (2 / 3, T0 + dt.timedelta(minutes=10))]:
    ref = cir[t]
    ref_f = np.where(ref.undetect, 0.0, np.clip(ref.values, 0, None))
    for method in ["nearest", "linear", "lk"]:
        val, cov, und = pair.get(alpha, method)
        out = np.where(und, 0.0, np.clip(val, 0, None))
        m = cov & ref.coverage & ((out > 10) | (ref_f > 10))
        rmse = np.sqrt(np.mean((out[m] - ref_f[m]) ** 2))
        print(f"{t:%H:%M}  alpha={alpha:.2f}  {method:8s} RMSE {rmse:5.2f} dBZ  ({m.sum()} px)")

## Next steps

- How the interpolation works and how it was validated: `docs/method.md`. Facts about the OPERA products: `docs/data.md`.
- Validation and production runs use the scripts: `docs/usage.md`. What is planned: `docs/roadmap.md`.
- New exploration? Copy this notebook as `NN-<initials>-<topic>.ipynb` (see `CONTRIBUTING.md`).